<a href="https://colab.research.google.com/github/gevargas/queerdatagap/blob/main/queerdatagap_clean_frequency_matrix_LGBTQ_vocabulary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Clean a frequency matrix and flag LGBTQ+ vocabulary

This notebook reads a frequency-matrix Excel file, keeps its first two columns (`term` and `stem`), removes a broad configurable set of Spanish stop words and duplicate terms, and creates a third column, `LGBTQ_related`, with:

- `1`: the term matches a configurable LGBTQ+/queer vocabulary lexicon or lexical pattern;
- `0`: no configured match was found.

**Important:** `LGBTQ_related = 1` is a research heuristic, not a claim about a speaker's identity and not a universal definition of queer vocabulary. The lexicon should be reviewed and adapted to the corpus, country, community, and research question.


In [ ]:

# Configuration
from pathlib import Path

INPUT_FILE = Path('/mnt/data/_rCcPtmnjpE_youtube_video_rccptmnjpe_frequency_matrix.xlsx')
OUTPUT_FILE = INPUT_FILE.with_name(INPUT_FILE.stem + '_clean_LGBTQ_flagged.xlsx')

# True = also remove conversational fillers, generic auxiliary/light verbs,
# courtesy expressions, numbers written as words, and other aggressive stop-word groups.
AGGRESSIVE_STOPWORDS = True

# True = include contextual vocabulary often involved in LGBTQ+ discourse
# (e.g. gender, sexuality, discrimination, rights, pronouns, marriage).
# False = mark only direct identity/community terms and strong lexical patterns.
BROAD_LGBTQ_CONTEXT = True

print('Input :', INPUT_FILE)
print('Output:', OUTPUT_FILE)


Input : /mnt/data/_rCcPtmnjpE_youtube_video_rccptmnjpe_frequency_matrix.xlsx
Output: /mnt/data/_rCcPtmnjpE_youtube_video_rccptmnjpe_frequency_matrix_clean_LGBTQ_flagged.xlsx


In [ ]:

import re
import unicodedata
import pandas as pd


def normalize(text):
    """Lowercase, strip whitespace, remove accents, and normalize internal spaces."""
    if pd.isna(text):
        return ''
    s = str(text).strip().lower()
    s = ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )
    s = re.sub(r'\s+', ' ', s)
    return s



## Spanish stop-word families

The lists below intentionally combine several families that are useful for cleaning oral/podcast transcriptions: articles and determiners, prepositions, conjunctions/connectors, pronouns, demonstratives, possessives, interrogatives/relatives, adverbs of place/time/degree, interjections, discourse fillers, courtesy expressions, auxiliary/light verbs, number words, and transcription artefacts.

Because stop-word removal is methodologically consequential, every group is explicit and editable.


In [ ]:

ARTICLES_DETERMINERS = {
    'el','la','los','las','un','una','unos','unas','lo','al','del'
}

PREPOSITIONS = {
    'a','ante','bajo','cabe','con','contra','de','desde','durante','en','entre',
    'hacia','hasta','mediante','para','por','segun','sin','so','sobre','tras','versus','via'
}

CONJUNCTIONS_CONNECTORS = {
    'y','e','ni','o','u','pero','mas','aunque','sino','si','porque','pues','como','que',
    'cuando','mientras','donde','adonde','entonces','ademas','tambien','tampoco','sin embargo',
    'por tanto','por eso','por ello','asi que','es decir','o sea','por ejemplo','en cambio',
    'incluso','inclusive','excepto','salvo','sea','bien','ya sea','luego'
}

PRONOUNS = {
    'yo','tu','usted','ustedes','vos','vosotros','vosotras','el','ella','elle','ellos','ellas','elles',
    'nosotros','nosotras','nosotres','me','te','se','nos','os','mi','ti','si','conmigo','contigo',
    'consigo','le','les','lo','la','los','las','ello','ellos','ellas','quien','quienes','cual','cuales',
    'algo','alguien','nadie','nada','cualquiera','cualesquiera','otro','otra','otros','otras'
}

DEMONSTRATIVES = {
    'este','esta','estos','estas','esto','ese','esa','esos','esas','eso',
    'aquel','aquella','aquellos','aquellas','aquello'
}

POSSESSIVES = {
    'mi','mis','mio','mia','mios','mias','tu','tus','tuyo','tuya','tuyos','tuyas','su','sus',
    'suyo','suya','suyos','suyas','nuestro','nuestra','nuestros','nuestras',
    'vuestro','vuestra','vuestros','vuestras'
}

INTERROGATIVES_RELATIVES = {
    'que','quien','quienes','cual','cuales','cuanto','cuanta','cuantos','cuantas',
    'como','cuando','donde','adonde','cuyo','cuya','cuyos','cuyas'
}

PLACE_ADVERBS = {
    'aqui','aca','ahi','alli','alla','cerca','lejos','arriba','abajo','adelante','delante',
    'atras','detras','encima','debajo','dentro','fuera','afuera','adentro','alrededor','enfrente'
}

TIME_ADVERBS = {
    'hoy','ayer','manana','ahora','antes','despues','luego','entonces','siempre','nunca','jamas',
    'ya','aun','todavia','pronto','tarde','temprano','actualmente','recientemente','anteriormente',
    'posteriormente','mientras','enseguida','inmediatamente','diariamente','semanalmente',
    'mensualmente','anualmente','frecuentemente','ocasionalmente'
}

DEGREE_QUANTITY_ADVERBS = {
    'muy','mucho','mucha','muchos','muchas','mas','menos','poco','poca','pocos','pocas','bastante',
    'bastantes','demasiado','demasiada','demasiados','demasiadas','tan','tanto','tanta','tantos',
    'tantas','casi','apenas','solo','solamente','unicamente','algo','nada','todo','toda','todos','todas'
}

INTERJECTIONS_FILLERS = {
    'ah','eh','em','mmm','mm','aja','ajá','ay','uy','uf','uff','oh','hey','hola','bueno','pues',
    'este','esto','digamos','sabes','sabe','verdad','vale','ok','okay','osea','onda','tipo',
    'basicamente','realmente','literalmente','simplemente','obviamente','claramente','exactamente',
    'efectivamente','finalmente','en fin','a ver','vamos','mira','miren','fijate','fijense'
}

COURTESY_META = {
    'gracias','muchas gracias','por favor','bienvenido','bienvenida','bienvenidos','bienvenidas',
    'buenos dias','buenas tardes','buenas noches','saludos','adios','chao','chau'
}

AUXILIARY_LIGHT_VERBS = {
    'ser','estar','haber','tener','hacer','poder','deber','ir','venir','decir','dar','poner','quedar',
    'seguir','llevar','pasar','ver','saber','querer','creer','pensar','parecer','hablar','decir',
    'dijo','dice','dicen','hacerlo','tenemos','tengo','tiene','tienen','vamos','voy','va','van',
    'puede','pueden','podemos','creo','pienso'
}

NUMBER_WORDS = {
    'cero','uno','una','dos','tres','cuatro','cinco','seis','siete','ocho','nueve','diez','once','doce',
    'trece','catorce','quince','dieciseis','diecisiete','dieciocho','diecinueve','veinte','cien','ciento',
    'mil','primero','primera','segundo','segunda','tercero','tercera'
}

TRANSCRIPTION_ARTEFACTS = {
    'musica','risas','risa','aplausos','aplauso','ruido','silencio','inaudible','ininteligible',
    'speaker','hablante','entrevistador','entrevistadora','entrevista','podcast','audio','video'
}

BASE_STOPWORDS = (
    ARTICLES_DETERMINERS | PREPOSITIONS | CONJUNCTIONS_CONNECTORS | PRONOUNS |
    DEMONSTRATIVES | POSSESSIVES | INTERROGATIVES_RELATIVES |
    PLACE_ADVERBS | TIME_ADVERBS | DEGREE_QUANTITY_ADVERBS
)

AGGRESSIVE_GROUPS = (
    INTERJECTIONS_FILLERS | COURTESY_META | AUXILIARY_LIGHT_VERBS |
    NUMBER_WORDS | TRANSCRIPTION_ARTEFACTS
)

STOPWORDS = {normalize(x) for x in BASE_STOPWORDS}
if AGGRESSIVE_STOPWORDS:
    STOPWORDS |= {normalize(x) for x in AGGRESSIVE_GROUPS}

print(f'{len(STOPWORDS):,} configured stop-word entries')


380 configured stop-word entries



## LGBTQ+/queer vocabulary heuristic

The strict lexicon contains direct identity/community terms. The optional broad contextual lexicon includes concepts frequently involved in LGBTQ+ discourse but that may also occur in non-LGBTQ+ contexts. Edit these sets to match the research design.


In [ ]:

DIRECT_LGBTQ_TERMS = {
    # umbrellas / communities
    'lgbt','lgbtq','lgbtq+','lgbti','lgbtiq','lgbtiq+','diversidad sexual','diversidad sexo generica',
    'queer','cuir','disidencia sexual','disidencias sexuales','disidencia sexogenerica',
    # sexual orientations
    'gay','gays','lesbiana','lesbianas','lesbico','lesbica','bisexual','bisexuales','pansexual',
    'pansexuales','asexual','asexuales','demisexual','demisexuales','homosexual','homosexuales',
    # gender identities / expressions
    'trans','transgenero','transgeneros','transexual','transexuales','travesti','travestis',
    'no binario','no binaria','no binaries','no binarie','no binarios','no binarias','nobinarie',
    'genero fluido','genderfluid','agenero','intersexual','intersexuales','intersex','cisgenero','cis',
    'drag','drag queen','drag king','transformista',
    # community / politics / prejudice
    'orgullo','pride','homofobia','homofobico','homofobica','lesbofobia','bifobia','transfobia',
    'transfobico','transfobica','interfobia','heteronorma','heteronormatividad','heteronormativo',
    'cisnorma','cisnormatividad','cisnormativo','cisheteronorma','cisheteronormatividad',
    'misgenero','deadname','deadnaming','salir del closet','closet','armario'
}

BROAD_CONTEXT_TERMS = {
    'genero','identidad','identidad de genero','expresion de genero','orientacion','orientacion sexual',
    'sexualidad','sexual','sexo','pronombre','pronombres','nombre social','identidad sexual',
    'diversidad','discriminacion','estigma','estigmatizacion','violencia','odio','crimen de odio',
    'derechos','derechos humanos','igualdad','equidad','inclusion','exclusion','visibilidad','invisibilidad',
    'matrimonio','matrimonio igualitario','pareja','parejas','adopcion','familia','familias',
    'comunidad','activismo','activista','militancia','colectivo','colectiva','orgullo',
    'salud sexual','salud reproductiva','vih','sida','prep','profilaxis','hormona','hormonas',
    'hormonizacion','transicion','transicionar','reasignacion','rectificacion','registro civil',
    'nombre','cuerpo','corporalidad','masculinidad','feminidad','masculino','femenino'
}

# Root patterns catch inflections and common variants in term/stem columns.
# Keep patterns specific enough to avoid obvious false positives.
LGBTQ_ROOT_PATTERNS = [
    r'^lgbt', r'^queer', r'^cuir$', r'^lesb', r'^bisex', r'^pansex', r'^asex', r'^demisex',
    r'^homosex', r'^trans(gener|sexual|fob|icion|vest|$)', r'^travest', r'^intersex',
    r'^nobin', r'^no binar', r'^agener', r'^cis(gener|norm|heter|$)', r'^heteronorm',
    r'^homofob', r'^lesbofob', r'^bifob', r'^transfob', r'^interfob', r'^sexogener',
    r'^pronombre', r'^hormoniz', r'^deadnam'
]
LGBTQ_ROOT_RE = re.compile('|'.join(f'(?:{p})' for p in LGBTQ_ROOT_PATTERNS))

DIRECT_NORM = {normalize(x) for x in DIRECT_LGBTQ_TERMS}
BROAD_NORM = {normalize(x) for x in BROAD_CONTEXT_TERMS}


def is_lgbtq_related(term, stem=''):
    t = normalize(term)
    s = normalize(stem)
    if t in DIRECT_NORM or LGBTQ_ROOT_RE.search(t) or (s and LGBTQ_ROOT_RE.search(s)):
        return 1
    if BROAD_LGBTQ_CONTEXT and t in BROAD_NORM:
        return 1
    return 0


In [ ]:

# Read and validate the input matrix
raw = pd.read_excel(INPUT_FILE)

if raw.shape[1] < 2:
    raise ValueError('The input workbook must contain at least two columns.')

# The request is to keep the first two columns, regardless of their names.
df = raw.iloc[:, :2].copy()
df.columns = ['term', 'stem']

print('Input rows:', len(df))
df.head(10)


Input rows: 3978


,term,stem
0,eh,eh
1,gracias,graci
2,pues,pues
3,deporte,deport
4,muchas,much
5,aqui,aqui
6,personas,person
7,bueno,buen
8,entonces,entonc
9,hacer,hac


In [ ]:

# Normalize only for cleaning/comparison; preserve the original term/stem text in the output.
df['_term_norm'] = df['term'].map(normalize)
df['_stem_norm'] = df['stem'].map(normalize)

# Remove empty terms, numeric-only tokens, URLs/emails, one-character noise, and configured stop words.
def is_noise(term_norm):
    if not term_norm:
        return True
    if term_norm in STOPWORDS:
        return True
    if len(term_norm) <= 1:
        return True
    if re.fullmatch(r'[\d.,:/_-]+', term_norm):
        return True
    if re.match(r'^(https?://|www\.)', term_norm):
        return True
    if '@' in term_norm and '.' in term_norm:
        return True
    return False

before = len(df)
df = df[~df['_term_norm'].map(is_noise)].copy()
after_stopwords = len(df)

# Remove duplicate terms accent- and case-insensitively, keeping the first occurrence.
df = df.drop_duplicates(subset=['_term_norm'], keep='first').copy()
after_duplicates = len(df)

# Add the requested third column.
df['LGBTQ_related'] = [
    is_lgbtq_related(t, s) for t, s in zip(df['term'], df['stem'])
]

# Final output: exactly the first two columns + the binary marker.
out = df[['term', 'stem', 'LGBTQ_related']].reset_index(drop=True)

print(f'Removed as stop words/noise: {before - after_stopwords:,}')
print(f'Removed as duplicate terms : {after_stopwords - after_duplicates:,}')
print(f'Final rows                 : {len(out):,}')
print(f'LGBTQ_related = 1          : {int(out.LGBTQ_related.sum()):,}')

out.head(25)


Removed as stop words/noise: 180
Removed as duplicate terms : 0
Final rows                 : 3,798
LGBTQ_related = 1          : 64


,term,stem,LGBTQ_related
0,deporte,deport,0
1,personas,person,0
2,mexico,mexic,0
3,ciudad,ciud,0
4,espacios,espaci,0
5,importante,import,0
6,lgbt,lgbt,1
7,comunidad,comun,1
8,discriminacion,discrimin,1
9,dia,dia,0


In [ ]:

# Review all terms marked as potentially LGBTQ+-related before exporting.
# This is an important manual-validation step for research use.
marked = out[out['LGBTQ_related'] == 1].copy()
marked


,term,stem,LGBTQ_related
6,lgbt,lgbt,1
7,comunidad,comun,1
8,discriminacion,discrimin,1
10,diversidad,divers,1
16,derechos,derech,1
...,...,...,...
3522,sexogenericas,sexogener,1
3666,transfobica,transfob,1
3681,travestie,travesti,1
3682,travestis,travestis,1


In [ ]:

# Export the cleaned three-column matrix.
out.to_excel(OUTPUT_FILE, index=False)
print(f'Saved: {OUTPUT_FILE}')


Saved: /mnt/data/_rCcPtmnjpE_youtube_video_rccptmnjpe_frequency_matrix_clean_LGBTQ_flagged.xlsx



## Recommended validation

Before using the binary flag analytically, review the marked terms in context. Generic terms such as `gender`, `rights`, `violence`, `family`, or `community` can be relevant to LGBTQ+ discourse but are not exclusively LGBTQ+ vocabulary. For a stricter extraction, set `BROAD_LGBTQ_CONTEXT = False` and rerun the notebook.

For corpus-specific refinement, add or remove terms from `DIRECT_LGBTQ_TERMS`, `BROAD_CONTEXT_TERMS`, and `LGBTQ_ROOT_PATTERNS` rather than hard-coding decisions elsewhere in the pipeline.
